<a href="https://colab.research.google.com/github/lahirua-madhusanka/data_warehouse_coursework/blob/dataset%2FstarSchema/Datawarehose_CW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**# ETL Pipeline for Fact and Dimension Tables**

ETL pipeline for loading a student habits dataset into Oracle Autonomous Data Warehouse using a dimensional model.**

In [5]:
!pip install oracledb pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.6 MB/s eta 0:00:00


In [6]:
import pandas as pd
import oracledb
import zipfile
import os
from datetime import datetime

**Upload the clean data set**

In [7]:
from google.colab import files
uploaded = files.upload()

Saving Cleaned_student_dataset.csv to Cleaned_student_dataset.csv


**Read the dataset**

In [8]:
df = pd.read_csv("Cleaned_student_dataset.csv")
print(df.shape)
df.head()

(998442, 42)


,study_hours,attendance,assignment_completion,midterm_score,final_score,project_score,backlogs,sleep_hours,stress,anxiety,...,hostel_student,extracurricular_hours,phone_unlocks_per_day,previous_gpa,class_participation,weekly_study_sessions,group_study_hours,financial_stress,gpa,performance_level
0,3.014684,67.00599,51.595387,57.211285,61.653540,65.397200,4,5.993893,4.287966,58.146000,...,0,1.958940,73.727480,5.721128,3.587111,2.814086,1.814086,5.491878,0.546729,Low
1,3.665277,73.28455,69.749020,57.552320,62.062782,65.715500,3,6.949383,1.841224,41.945290,...,0,3.146447,48.468456,5.755232,4.820090,2.836821,1.836821,2.876881,0.707133,Low
2,2.703784,72.32519,92.837640,44.568970,46.482760,53.597702,2,6.703293,3.863112,56.555750,...,1,5.551245,46.623684,4.456897,5.493774,1.971265,0.971265,5.704047,0.868230,Low
3,3.445073,74.75687,85.189026,52.040790,55.448948,60.571404,2,6.498832,5.073206,65.171420,...,1,4.543216,47.909600,5.204079,5.481987,2.469386,1.469386,6.596658,0.729216,Low
4,0.192687,55.05021,64.520620,32.815000,32.378000,42.627330,5,6.552570,1.000000,30.725826,...,1,4.447042,73.316520,3.281500,2.822375,1.187667,0.187667,4.602954,0.370964,Low


**Standardize the column names**

In [9]:
df.columns = df.columns.str.lower().str.strip()
df.columns = df.columns.str.replace(" ", "_")
df.columns = df.columns.str.replace(r'[^a-z0-9_]', '', regex=True)
print(df.columns)

Index(['study_hours', 'attendance', 'assignment_completion', 'midterm_score',
       'final_score', 'project_score', 'backlogs', 'sleep_hours', 'stress',
       'anxiety', 'depression', 'motivation', 'concentration',
       'time_management', 'self_discipline', 'social_media_hours',
       'gaming_hours', 'netflix_hours', 'screen_time', 'physical_activity',
       'junk_food_frequency', 'caffeine_mg', 'late_night_frequency',
       'procrastination_score', 'family_income', 'parental_education_level',
       'internet_quality', 'library_visits', 'online_courses_completed',
       'part_time_hours', 'peer_study_group', 'relationship_status',
       'hostel_student', 'extracurricular_hours', 'phone_unlocks_per_day',
       'previous_gpa', 'class_participation', 'weekly_study_sessions',
       'group_study_hours', 'financial_stress', 'gpa', 'performance_level'],
      dtype='object')


**Genarate Student Code**

In [10]:
df = df.reset_index(drop=True)
df["student_code"] = ["STD" + str(i+1).zfill(7) for i in range(len(df))]

**Convert Numaric Column**

In [11]:
numeric_cols = [
    'study_hours','attendance','assignment_completion','midterm_score','final_score',
    'project_score','backlogs','sleep_hours','stress','anxiety','depression',
    'motivation','concentration','time_management','self_discipline',
    'social_media_hours','gaming_hours','netflix_hours','screen_time',
    'physical_activity','junk_food_frequency','caffeine_mg','late_night_frequency',
    'procrastination_score','family_income','library_visits','online_courses_completed',
    'part_time_hours','extracurricular_hours','phone_unlocks_per_day','previous_gpa',
    'class_participation','weekly_study_sessions','group_study_hours',
    'financial_stress','gpa'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

**Remove Invalid column from import column**

In [12]:
important_cols = ['study_hours', 'attendance', 'gpa', 'performance_level']
df = df.dropna(subset=important_cols)
print(df.shape)

(998442, 43)


**Create all the diamention dataframes**

**dim_student**

In [13]:
dim_student = df[[
    'student_code',
    'family_income',
    'parental_education_level',
    'internet_quality',
    'relationship_status',
    'hostel_student',
    'peer_study_group'
]].copy()

dim_student = dim_student.drop_duplicates(subset=['student_code'])
print(dim_student.shape)
dim_student.head()

(998442, 7)


,student_code,family_income,parental_education_level,internet_quality,relationship_status,hostel_student,peer_study_group
0,STD0000001,20107.828,3,5.051189,1,0,0
1,STD0000002,47697.250,4,6.586138,0,0,0
2,STD0000003,35155.830,2,4.733003,0,1,0
3,STD0000004,23898.123,2,4.414770,0,1,0
4,STD0000005,22090.764,2,4.112101,1,1,0


**dim_performnce level**

In [14]:
dim_performance = df[['performance_level']].drop_duplicates().copy()
dim_performance = dim_performance.reset_index(drop=True)
dim_performance.head()

,performance_level
0,Low


**dim_load_date**

In [15]:
today = datetime.today()
date_key = int(today.strftime("%Y%m%d"))

dim_date = pd.DataFrame([{
    "date_key": date_key,
    "full_date": today.date(),
    "day_number": today.day,
    "month_number": today.month,
    "year_number": today.year
}])

dim_date

,date_key,full_date,day_number,month_number,year_number
0,20260321,2026-03-21,21,3,2026


**Upload the oracle wallte**

In [16]:
uploaded = files.upload()

Saving Wallet_studentdw.zip to Wallet_studentdw.zip


**Extract the wallet**

In [17]:
wallet_zip = "Wallet_studentdw.zip"
wallet_dir = "/content/wallet"

os.makedirs(wallet_dir, exist_ok=True)

with zipfile.ZipFile(wallet_zip, 'r') as zip_ref:
    zip_ref.extractall(wallet_dir)

print("Wallet extracted successfully")

Wallet extracted successfully


**connect to oracle ADW**

In [18]:
username = "ADMIN"
password = "Lm@Madush$96"
dsn = "studentdw_high"

connection = oracledb.connect(
    user=username,
    password=password,
    dsn=dsn,
    config_dir=wallet_dir,
    wallet_location=wallet_dir,
    wallet_password=password
)

cursor = connection.cursor()
print("Connected to Oracle ADW successfully")

Connected to Oracle ADW successfully


**Load Diamention table**

**Load dim_performance_level**

In [19]:
cursor.execute("TRUNCATE TABLE dim_performance_level")

perf_rows = [tuple(x) for x in dim_performance.to_numpy()]
cursor.executemany(
    "INSERT INTO dim_performance_level (performance_level) VALUES (:1)",
    perf_rows
)

connection.commit()
print("dim_performance_level loaded")

dim_performance_level loaded


*load dim_load_data**

In [20]:
cursor.execute("DELETE FROM dim_load_date WHERE date_key = :1", [date_key])

cursor.execute("""
    INSERT INTO dim_load_date (date_key, full_date, day_number, month_number, year_number)
    VALUES (:1, :2, :3, :4, :5)
""", [
    int(dim_date.loc[0, "date_key"]),
    dim_date.loc[0, "full_date"],
    int(dim_date.loc[0, "day_number"]),
    int(dim_date.loc[0, "month_number"]),
    int(dim_date.loc[0, "year_number"])
])

connection.commit()
print("dim_load_date loaded")

dim_load_date loaded


**load dim_student**

In [21]:
cursor.execute("TRUNCATE TABLE dim_student")

student_rows = [tuple(x) for x in dim_student.to_numpy()]
cursor.executemany("""
    INSERT INTO dim_student (
        student_code, family_income, parental_education_level,
        internet_quality, relationship_status, hostel_student, peer_study_group
    ) VALUES (:1, :2, :3, :4, :5, :6, :7)
""", student_rows)

connection.commit()
print("dim_student loaded")

dim_student loaded


**Read surrogate keys back from the database**

read student key

In [22]:
cursor.execute("SELECT student_key, student_code FROM dim_student")

student_key_data = cursor.fetchall()

import pandas as pd
student_key_map = pd.DataFrame(student_key_data, columns=["student_key", "student_code"])

print(student_key_map.shape)
student_key_map.head()

(998442, 2)


,student_key,student_code
0,1014660,STD0016218
1,1014661,STD0016219
2,1014662,STD0016220
3,1014663,STD0016221
4,1014664,STD0016222


In [26]:
insert_fact_sql = """INSERT INTO fact_student_performance (
    student_key, performance_key, date_key, study_hours, attendance,
    assignment_completion, midterm_score, final_score, project_score, backlogs,
    sleep_hours, stress, anxiety, depression, motivation, concentration,
    time_management, self_discipline, social_media_hours, gaming_hours,
    netflix_hours, screen_time, physical_activity, junk_food_frequency,
    caffeine_mg, late_night_frequency, procrastination_score, library_visits,
    online_courses_completed, part_time_hours, extracurricular_hours,
    phone_unlocks_per_day, previous_gpa, class_participation,
    weekly_study_sessions, group_study_hours, financial_stress, gpa
) VALUES (
    :1, :2, :3, :4, :5, :6, :7, :8, :9, :10, :11, :12, :13, :14, :15, :16, :17,
    :18, :19, :20, :21, :22, :23, :24, :25, :26, :27, :28, :29, :30, :31, :32,
    :33, :34, :35, :36, :37, :38
)"""

**Verify the content of the key maps after running the previous modified cells**

In [28]:
cursor.execute("SELECT performance_key, performance_level FROM dim_performance_level")

performance_key_data = cursor.fetchall()

performance_key_map = pd.DataFrame(
    performance_key_data,
    columns=["performance_key", "performance_level"]
)

print('performance_key_map shape:', performance_key_map.shape)
performance_key_map.head()

performance_key_map shape: (1, 2)


,performance_key,performance_level
0,2,Low


In [29]:
print('student_key_map shape:', student_key_map.shape)
display(student_key_map.head())

print('performance_key_map shape:', performance_key_map.shape)
display(performance_key_map.head())

student_key_map shape: (998442, 2)


,student_key,student_code
0,1014660,STD0016218
1,1014661,STD0016219
2,1014662,STD0016220
3,1014663,STD0016221
4,1014664,STD0016222


performance_key_map shape: (1, 2)


,performance_key,performance_level
0,2,Low


read prforance key

In [30]:
cursor.execute("SELECT performance_key, performance_level FROM dim_performance_level")
performance_key_data = cursor.fetchall()
performance_key_map = pd.DataFrame(performance_key_data, columns=["performance_key", "performance_level"])

print(performance_key_map.shape)
performance_key_map.head()

(1, 2)


,performance_key,performance_level
0,2,Low


**Build the fact DataFrame**

In [31]:
fact_df = df.merge(student_key_map, on="student_code", how="left")
fact_df = fact_df.merge(performance_key_map, on="performance_level", how="left")
fact_df["date_key"] = date_key

In [32]:
fact_df = fact_df[[
    'student_key', 'performance_key', 'date_key',
    'study_hours','attendance','assignment_completion','midterm_score',
    'final_score','project_score','backlogs','sleep_hours','stress',
    'anxiety','depression','motivation','concentration','time_management',
    'self_discipline','social_media_hours','gaming_hours','netflix_hours',
    'screen_time','physical_activity','junk_food_frequency','caffeine_mg',
    'late_night_frequency','procrastination_score','library_visits',
    'online_courses_completed','part_time_hours','extracurricular_hours',
    'phone_unlocks_per_day','previous_gpa','class_participation',
    'weekly_study_sessions','group_study_hours','financial_stress','gpa'
]]

print(fact_df.shape)
fact_df.head()

(998442, 38)


,student_key,performance_key,date_key,study_hours,attendance,assignment_completion,midterm_score,final_score,project_score,backlogs,...,online_courses_completed,part_time_hours,extracurricular_hours,phone_unlocks_per_day,previous_gpa,class_participation,weekly_study_sessions,group_study_hours,financial_stress,gpa
0,998443,2,20260321,3.014684,67.00599,51.595387,57.211285,61.653540,65.397200,4,...,0,5.872288,1.958940,73.727480,5.721128,3.587111,2.814086,1.814086,5.491878,0.546729
1,998444,2,20260321,3.665277,73.28455,69.749020,57.552320,62.062782,65.715500,3,...,0,4.586602,3.146447,48.468456,5.755232,4.820090,2.836821,1.836821,2.876881,0.707133
2,998445,2,20260321,2.703784,72.32519,92.837640,44.568970,46.482760,53.597702,2,...,1,5.359920,5.551245,46.623684,4.456897,5.493774,1.971265,0.971265,5.704047,0.868230
3,998446,2,20260321,3.445073,74.75687,85.189026,52.040790,55.448948,60.571404,2,...,1,4.926094,4.543216,47.909600,5.204079,5.481987,2.469386,1.469386,6.596658,0.729216
4,998447,2,20260321,0.192687,55.05021,64.520620,32.815000,32.378000,42.627330,5,...,0,7.478159,4.447042,73.316520,3.281500,2.822375,1.187667,0.187667,4.602954,0.370964


convert rows to tuples

In [33]:
fact_rows = [tuple(x) for x in fact_df.to_numpy()]
print("Total fact rows:", len(fact_rows))

Total fact rows: 998442


insert in batch

In [34]:
batch_size = 10000

for i in range(0, len(fact_rows), batch_size):
    batch = fact_rows[i:i+batch_size]
    cursor.executemany(insert_fact_sql, batch)
    connection.commit()
    print(f"Inserted rows {i} to {i + len(batch)}")

Inserted rows 0 to 10000
Inserted rows 10000 to 20000
Inserted rows 20000 to 30000
Inserted rows 30000 to 40000
Inserted rows 40000 to 50000
Inserted rows 50000 to 60000
Inserted rows 60000 to 70000
Inserted rows 70000 to 80000
Inserted rows 80000 to 90000
Inserted rows 90000 to 100000
Inserted rows 100000 to 110000
Inserted rows 110000 to 120000
Inserted rows 120000 to 130000
Inserted rows 130000 to 140000
Inserted rows 140000 to 150000
Inserted rows 150000 to 160000
Inserted rows 160000 to 170000
Inserted rows 170000 to 180000
Inserted rows 180000 to 190000
Inserted rows 190000 to 200000
Inserted rows 200000 to 210000
Inserted rows 210000 to 220000
Inserted rows 220000 to 230000
Inserted rows 230000 to 240000
Inserted rows 240000 to 250000
Inserted rows 250000 to 260000
Inserted rows 260000 to 270000
Inserted rows 270000 to 280000
Inserted rows 280000 to 290000
Inserted rows 290000 to 300000
Inserted rows 300000 to 310000
Inserted rows 310000 to 320000
Inserted rows 320000 to 330000


Verify the load

In [35]:
cursor.execute("SELECT COUNT(*) FROM dim_student")
print("dim_student:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM dim_performance_level")
print("dim_performance_level:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM dim_load_date")
print("dim_load_date:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM fact_student_performance")
print("fact_student_performance:", cursor.fetchone()[0])

dim_student: 998442
dim_performance_level: 1
dim_load_date: 1
fact_student_performance: 998442


Close cdonnection

In [36]:
cursor.close()
connection.close()
print("ETL completed successfully")

ETL completed successfully
